# Running the typed decision models

`SMOKE = True` sends a small sample to every model in every format: `SMOKE_ITEMS` items per test (default 1), with
all their variants and original prompts. With 1 item that is 56 requests per model and format, about 1,500 in
total. It checks that the code works and that every model accepts our requests.
`SMOKE = False` runs the full design: 14,695 requests per model and format.

Answers are cached, so a re-run only sends what is missing. Results go to `data/answers/tdms/`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
SMOKE = False
SMOKE_ITEMS = 1
CONCURRENCY = 32      # requests in flight per model; the client backs off by itself on rate limits
# Mercury Decide is only offered as a free variant, which OpenRouter limits to 1,000 requests per day
# (HTTP 429 "free-models-per-day-high-balance", 2026-10-04). Its first 1,026 answers are kept; whether to
# drop it or reach it another way is decided with the author.
SKIP = ["inception/mercury-decide:free"]

In [3]:
import sys
from pathlib import Path
CODE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE))   # judgebias and tdm
import pandas as pd
from judgebias import authority, design, run

rows = design.build_design(citations=authority.load_citations())
print("differences from Table 2 of the paper:", design.check_counts(rows) or "none")
assert SMOKE or not design.check_counts(rows), "the full run needs all 900 citations: run 02_citations with SMOKE = False first"
if SMOKE:
    rows = design.smoke_sample(rows, n_items=SMOKE_ITEMS)
n_requests = len(run.requests(rows, "calm"))
print(f"{n_requests} requests per model and format, {n_requests * len(run.MODELS) * len(run.FORMATS)} in total")
pd.Series(design.counts(rows), name="prompts")

differences from Table 2 of the paper: none
14695 requests per model and format, 396765 in total


original        972
position        439
bandwagon      1756
distraction     878
diversity      6146
compassion      439
authority       900
verbosity       500
fallacy         500
sentiment      1193
Name: prompts, dtype: int64

## One example per bias: the CALM prompt string the request is built from

In [4]:
for bias in design.EXPECTED:
    r = next(x for x in rows if x["bias"] == bias)
    print("=" * 30, bias, r["variant"], "| R1 shown as", r["r1_label"], "| target", r["target"], "=" * 30)
    print(r["prompt"].strip()[-700:])        # the end of the prompt, where the perturbation usually is

============================== original - | R1 shown as B | target None ==============================
he others—the ones in the cold place where Sarah Palin is from, and that tropical paradise where that TV show Lost was filmed, I think? We've got those obvious ones like Texas, California, New York, and Florida, but also a bunch of smaller ones all smooshed together on the east coast. They add up quickly, and there's a nice round number to it. It's not too few that it seems like an exclusive club, but it's not like we're trying to hit a century or anything. It's a number perfect for a set of collectible quarters, two for each hand times the fingers on a hand, making that total... oh, right, that familiar even number we learn as kids – the total is 50 states.
[The End of Assistant B’s Answer]
============================== position swapped | R1 shown as A | target R2 ==============================
+ 14 = 56, 56 + 16 = 72, 72 + 18 = 90, and finally, 90 + 20 = 110. Since we have calculat

## Run every model in every format

In [5]:
summary, clients = [], {}
for model in run.MODELS:
    if model in SKIP:
        continue
    for fmt in run.FORMATS:
        records, client = run.run_model(model, fmt, rows, smoke=SMOKE, budget_usd=10, concurrency=CONCURRENCY)
        clients[(model, fmt)] = client
        ok = [r for r in records if r["ok"]]
        summary.append({"model": model, "format": fmt, "requests": len(records), "ok": len(ok),
                        "failed": len(records) - len(ok), "ties": sum(r["choice"] is None for r in ok),
                        "usd": sum(r["cost"] or 0 for r in records),
                        "tokens_in": sum(r["input_tokens"] or 0 for r in records),
                        "median_latency_s": pd.Series([r["latency_s"] for r in ok if not r["cached"]]).median(),
                        "versions": sorted({r["snapshot"] for r in ok})})
summary = pd.DataFrame(summary)
summary

typesafe/jev-1.13 [calm]: 100%|##########| 14695/14695 [00:00<?, ?it/s]

typesafe/jev-1.13 [native]: 100%|##########| 14695/14695 [00:00<?, ?it/s]

typesafe/jev-1.13 [decomposed]: 100%|##########| 14695/14695 [00:00<?, ?it/s]

jaredpalmer/kev-4b [calm]: 100%|##########| 14695/14695 [00:00<?, ?it/s]

jaredpalmer/kev-4b [native]: 100%|##########| 14695/14695 [00:00<?, ?it/s]

...

cloudflare/clef [decomposed]:   0%|          | 56/14695 [00:00<?, ?it/s]

cloudflare/clef-flash [calm]:   0%|          | 56/14695 [00:00<?, ?it/s]

cloudflare/clef-flash [native]:   0%|          | 56/14695 [00:00<?, ?it/s]

cloudflare/clef-flash [decomposed]:   0%|          | 56/14695 [00:00<?, ?it/s]

,model,format,requests,ok,failed,ties,usd,tokens_in,median_latency_s,versions
0,typesafe/jev-1.13,calm,14695,14695,0,0,0.656385,15628208,NaN,[typesafe/jev-1.13-20260917]
1,typesafe/jev-1.13,native,14695,14695,0,0,0.662680,15778106,NaN,[typesafe/jev-1.13-20260917]
2,typesafe/jev-1.13,decomposed,14695,14695,0,16,1.188681,28301931,NaN,[typesafe/jev-1.13-20260917]
3,jaredpalmer/kev-4b,calm,14695,14695,0,0,0.479565,11418226,NaN,[jaredpalmer/kev-4b-20260924]
4,jaredpalmer/kev-4b,native,14695,14695,0,0,0.465658,11087105,NaN,[jaredpalmer/kev-4b-20260924]
5,jaredpalmer/kev-4b,decomposed,14695,14695,0,2,0.865133,20598410,NaN,[jaredpalmer/kev-4b-20260924]
6,togethercomputer/tev1-4b-experimental,calm,14695,14694,1,0,0.531232,12648373,NaN,[togethercomputer/tev1-4b-experimental-20260923]
7,togethercomputer/tev1-4b-experimental,native,14695,14695,0,0,0.514735,12255598,NaN,[togethercomputer/tev1-4b-experimental-20260923]
8,togethercomputer/tev1-4b-experimental,decomposed,14695,14695,0,31,3.027309,72078783,NaN,[togethercomputer/tev1-4b-experimental-20260923]
9,upstage/solar-decide,calm,14695,14695,0,0,0.811007,16220139,2.973272,[upstage/solar-decide-20260928]


In [6]:
# Any failures? Show the first failed exchange per model and format, exactly as sent and received.
for (model, fmt), client in clients.items():
    failed = [r for r in run.load_records(model, fmt, SMOKE) if not r["ok"]]
    if failed:
        print(f"{model} [{fmt}]: {len(failed)} failed, e.g. {failed[0]['error']}")

togethercomputer/tev1-4b-experimental [calm]: 1 failed, e.g. HTTP 502: {"error":{"message":"Tev1 answered \"D\" for question \"verdict\"","code":502}}


perplexity/pplx-decider-v1-27b [calm]: 5853 failed, e.g. HTTP 429: {"error":{"message":"HTTP 429: {\"error\":{\"code\":null,\"message\":\"Request rate limit exceeded, please try again later.\",\"param\":null,\"type\":\"too_many_requests\"}}\n","code":429}}
perplexity/pplx-decider-v1-27b [native]: 5687 failed, e.g. HTTP 429: {"error":{"message":"HTTP 429: {\"error\":{\"code\":null,\"message\":\"Request rate limit exceeded, please try again later.\",\"param\":null,\"type\":\"too_many_requests\"}}\n","code":429}}


perplexity/pplx-decider-v1-27b [decomposed]: 5544 failed, e.g. HTTP 429: {"error":{"message":"HTTP 429: {\"error\":{\"code\":null,\"message\":\"Request rate limit exceeded, please try again later.\",\"param\":null,\"type\":\"too_many_requests\"}}\n","code":429}}


## Notes

**Smoke run, 2026-10-04 (1 item per test, 56 requests per model and format, 1,512 in total).**
* All 1,512 requests were answered, with no failure and no retry. Every decomposed answer has its six
  sub-answers, no answer is an exact tie, and every probability lies in [0, 1].
* Every model accepts all three formats, including the six-question request of the decomposed format
  (a worry for Tev1, which its description calls a single-choice model).
* Seven models report a dated version (e.g. `typesafe/jev-1.13-20260917`, `jaredpalmer/kev-4b-20260924`).
  The two Cloudflare models report only `cloudflare/clef` and `cloudflare/clef-flash`.
* Sanity of the bookkeeping: on the nine original prompts the models pick the better response in 67 to
  89% of cases (a mix-up of A and B would give about 50%), and they repeat their verdict on the second
  sending, except Solar once in the CALM format (Solar was already known to be noisy).
* Speed (median seconds per request): Jev 0.3, Tev1 0.3 to 0.6, Mercury 0.3 to 0.5, Perplexity 0.5 to 1.0,
  Clef 0.7 to 1.1, D1 0.6 to 1.1, Kev 0.9 to 1.0, Clef Flash 0.7 to 1.7, Solar 1.2 to 8.2.
* Cost: $0.13 in total, so the full run costs about $35. At 8 requests at a time and one model after another
  it would take about 15 hours, 6.4 of them for Solar.

**Full run, 2026-10-04 (8 models x 3 formats x 14,695 requests).**
* Started at about 12:00 with 32 parallel requests per model. Restarted at 13:41 with Mercury Decide skipped
  (see below); the models already done were replayed from the cache in seconds.
* Every model answered all 44,085 of its requests, except one Tev1 request: for one authority prompt Tev1 answers
  the option "D", which does not exist (OpenRouter returns HTTP 502 `Tev1 answered "D"`). It counts as a missing
  verdict. The question text of that item lists its own options A to D, which Tev1 seems to confuse with ours.
* Each model kept one version for the whole run: `jev-1.13-20260917`, `kev-4b-20260924`,
  `tev1-4b-experimental-20260923`, `solar-decide-20260928`, `d1-20260930`, `pplx-decider-v1-27b-20261001`; the two
  Cloudflare models report no dated version (`cloudflare/clef`, `cloudflare/clef-flash`).
* **Mercury Decide** is only offered as a free variant, which OpenRouter limits to 1,000 requests per day (HTTP 429
  `free-models-per-day-high-balance`). It stopped after 1,026 answers in the CALM format and is left out of the
  analysis; a decision on dropping it is pending.
* **Perplexity**'s provider rate-limited 32 parallel requests (HTTP 429 "Request rate limit exceeded"): 8,842, 9,008
  and 9,151 of 14,695 were answered in the three formats. A second pass at 4 parallel requests answered the rest
  (with many retries), so Perplexity is complete as well.
* Time and cost per model: see `04_analysis` (time and cost section) and `outputs/tables/time_and_cost.csv`.

In [ ]:
# The released files are the data/ folder of this repository.